# 7.01 VLM 是怎么"看图读字"的

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/07_vla/7.01_vlm_basics.ipynb)


## 为什么要学

VLA = 预训练的 VLM（视觉-语言模型）+ 动作输出。π0 的 VLM 是 **PaliGemma-3B**：SigLIP 视觉塔 + Gemma-2B 语言模型。理解 VLM 怎么把"图像 + 文字"变成一串 token，是读懂 π0 的第一步。

本课对照 openpi 的 `embed_prefix`、`tokenizer.py`、`image_tools.py`，把前缀（prefix）的构造过程完整走一遍，并在 mini-π0 上动手实现。

---

## 1. PaliGemma 的结构

```text
  相机图像 (224×224×3)
     └─ SigLIP So400m/14 视觉塔（27 层 ViT，宽 1152）──> 256 个图像 token (256, 1152)
     └─ multi_modal_projector（一个线性层 1152 -> 2048）──> (256, 2048)
  文字："Task: pick up the cable, State: 128 64 ...;\nAction: "
     └─ SentencePiece 分词（词表 257152）──> token id
     └─ embed_tokens（257152 × 2048）× sqrt(2048) ──> (L_text, 2048)
  拼接 [图像 token..., 文字 token...]（前缀，内部双向注意力）
     └─ Gemma-2B（18 层，宽 2048）
```

ALOHA 有 3 个相机（顶部 + 左右腕部），π0 对每个相机都过一遍 SigLIP，得到 3×256=768 个图像 token。**图像 token 占了前缀的绝大部分**，所以 VLA 推理时前缀的计算量很大，必须用 KV cache 只算一次。

> RLinf 中 RLT 的 `prefix_seq_len=768`，正是 3 个相机 × 256 个 token。RL token 编码器就是从这 768 个前缀 embedding 里"压缩"出一个 token（第 8 章）。

## 2. 图像预处理：`resize_with_pad`

不同相机分辨率不同（ALOHA 是 640×480）。openpi 先**保持长宽比**缩放，再用黑边**填充**成 224×224（复刻 `tf.image.resize_with_pad`），最后归一化到 $[-1,1]$。直接拉伸会改变物体形状，对精细操作不利。

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from vla_ladder.envs import ToyTableEnv, CharTokenizer
from vla_ladder.models.mini_pi0 import MiniPi0, MiniPi0Config
from vla_ladder.models.transformer import make_att_2d_masks
from vla_ladder.utils import set_seed

### 练习 1：实现 `resize_with_pad`（PyTorch 版）

In [ ]:
def resize_with_pad(images, height, width):
    """images: (B, C, H, W) float -> (B, C, height, width)。保持长宽比缩放，四周补 0（黑边），居中放置"""
    B, C, H, W = images.shape
    # TODO（与 openpi 的写法一致）：
    # ratio = max(W / width, H / height)；resized = int(H / ratio), int(W / ratio)
    # 用 F.interpolate(..., mode="bilinear", align_corners=False) 缩放
    # pad_h0, rem_h = divmod(height - rh, 2)；pad_w0, rem_w = divmod(width - rw, 2)
    # F.pad(x, (pad_w0, pad_w0 + rem_w, pad_h0, pad_h0 + rem_h))
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def resize_with_pad(images, height, width):
    """images: (B, C, H, W) float -> (B, C, height, width)。保持长宽比缩放，四周补 0（黑边），居中放置"""
    B, C, H, W = images.shape
    ratio = max(W / width, H / height)
    rh, rw = int(H / ratio), int(W / ratio)
    x = F.interpolate(images, size=(rh, rw), mode="bilinear", align_corners=False)
    pad_h0, rem_h = divmod(height - rh, 2)
    pad_w0, rem_w = divmod(width - rw, 2)
    return F.pad(x, (pad_w0, pad_w0 + rem_w, pad_h0, pad_h0 + rem_h))

In [ ]:
# ✅ 自测：640x480 的 ALOHA 相机图像 -> 224x224
img = torch.rand(1, 3, 480, 640)
out = resize_with_pad(img, 224, 224)
assert out.shape == (1, 3, 224, 224)
assert out[0, :, :27].abs().sum() == 0 and out[0, :, -27:].abs().sum() == 0     # 上下各有约 28 行黑边
assert out[0, :, 112].abs().sum() > 0
print("通过 ✅ 有效区域高度:", int(480 / (640 / 224)), "像素")
plt.figure(figsize=(3, 3)); plt.imshow(out[0].permute(1, 2, 0)); plt.title("resize_with_pad 640x480 -> 224x224"); plt.show()

## 3. 提示词格式：π0 vs π0.5

对照 openpi `tokenizer.py` 中的 `PaligemmaTokenizer.tokenize`：

| | 格式 | 状态在哪里 |
|---|---|---|
| π0 | `"<bos>pick up the cable\n"` | 不在文字里，作为一个**连续向量 token** 放在后缀（动作专家那边） |
| π0.5 | `"<bos>Task: pick up the cable, State: 131 77 200 ...;\nAction: "` | **离散化成 256 个 bin 的整数**，写成文字放进前缀 |

π0.5 把状态写成文字，是为了让 VLM 能"读到"状态，也为了配合"离散 FAST 动作 token + 连续流匹配"的联合训练（知识隔离，7.02 课）。

In [ ]:
state = np.array([-1.0, -0.2, 0.5, 0.99])
bins = np.digitize(state, bins=np.linspace(-1, 1, 256 + 1)[:-1]) - 1
print("π0.5 的状态文字:", "State: " + " ".join(map(str, bins)) + ";")

我们的玩具任务用最简单的**字符级分词器**（`vla_ladder.envs.CharTokenizer`，一个汉字一个 token），指令如"把夹爪移到蓝色方块上"。

In [ ]:
tok = CharTokenizer()
ids, mask = tok.encode(["去红色方块", "把夹爪移到蓝色方块上"], max_len=16)
print("词表大小:", tok.vocab_size)
print("token id:\n", ids)
print("mask（True = 真实 token，False = padding）:\n", mask.astype(int))

---

## 4. `embed_prefix`：把图像和文字拼成前缀

openpi 的 `PI0Pytorch.embed_prefix`（节选，已简化）：

```py
for img, img_mask in zip(images, img_masks):              # 每个相机
    img_emb = self.paligemma_with_expert.embed_image(img)   # SigLIP + projector -> (B, 256, 2048)
    embs.append(img_emb)
    pad_masks.append(img_mask[:, None].expand(bsize, num_img_embs))   # 相机缺失时整组 token 都是 padding
    att_masks += [0] * num_img_embs                         # 0：与前面同属一个块（双向）
lang_emb = self.paligemma_with_expert.embed_language_tokens(lang_tokens)
lang_emb = lang_emb * math.sqrt(lang_emb_dim)               # Gemma 的惯例：embedding 乘 sqrt(宽度)
embs.append(lang_emb); pad_masks.append(lang_masks)
att_masks += [0] * num_lang_embs                            # 图像和文字之间也是双向的
```

两个细节值得注意：

1. **`img_mask`**：如果某个相机这一帧没有图像（比如单臂任务没有右腕相机），整组图像 token 标记为 padding，不会被任何 token 注意到。LIBERO、DROID 等不同数据集相机数量不同，就是靠这个统一的。
2. **`att_masks` 全是 0**：前缀内部是**完全双向**的，和 PaliGemma 预训练时一样，这样才能直接复用预训练权重。

### 练习 2：多相机前缀

mini-π0 的视觉塔一次只处理一张图。实现一个支持**多个相机**（其中某些可能缺失）的 `embed_prefix_multi`，返回 `(embs, pad_masks, att_masks)`。

In [ ]:
set_seed(0)
cfg = MiniPi0Config(vocab_size=tok.vocab_size)
model = MiniPi0(cfg)

def embed_prefix_multi(model, images_list, img_masks, lang_tokens, lang_masks):
    """
    images_list: 长度为 n_cam 的列表，每个 (B, 3, 48, 48)
    img_masks:   长度为 n_cam 的列表，每个 bool (B,)：这个相机这一帧是否有效
    返回 embs (B, n_cam*36 + L, W)、pad_masks bool (B, ·)、att_masks long (B, ·)
    """
    embs, pads = [], []
    for img, m in zip(images_list, img_masks):
        # TODO: e = model.vision(img)；pads 里放 m[:, None].expand(B, e.shape[1])
        raise NotImplementedError
    # TODO: 文字：model.embed_tokens(lang_tokens) * sqrt(vlm_width)；pads 放 lang_masks
    # TODO: 拼接；att_masks 全 0
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
set_seed(0)
cfg = MiniPi0Config(vocab_size=tok.vocab_size)
model = MiniPi0(cfg)

def embed_prefix_multi(model, images_list, img_masks, lang_tokens, lang_masks):
    """
    images_list: 长度为 n_cam 的列表，每个 (B, 3, 48, 48)
    img_masks:   长度为 n_cam 的列表，每个 bool (B,)：这个相机这一帧是否有效
    返回 embs (B, n_cam*36 + L, W)、pad_masks bool (B, ·)、att_masks long (B, ·)
    """
    embs, pads = [], []
    for img, m in zip(images_list, img_masks):
        e = model.vision(img)
        embs.append(e)
        pads.append(m[:, None].expand(e.shape[0], e.shape[1]))
    embs.append(model.embed_tokens(lang_tokens) * math.sqrt(model.cfg.vlm_width))
    pads.append(lang_masks)
    embs, pads = torch.cat(embs, 1), torch.cat(pads, 1)
    return embs, pads, torch.zeros_like(pads, dtype=torch.long)

In [ ]:
# ✅ 自测：两个相机，第 2 个样本的腕部相机缺失
env = ToyTableEnv(num_envs=2, seed=0); obs = env.reset()
imgs = torch.tensor(obs["image"])
ids, msk = tok.encode(obs["instruction"])
cam_masks = [torch.tensor([True, True]), torch.tensor([True, False])]
embs, pad, att = embed_prefix_multi(model, [imgs, imgs], cam_masks, torch.tensor(ids), torch.tensor(msk))
assert embs.shape == (2, 36 * 2 + 16, cfg.vlm_width)
assert pad[1, 36:72].sum() == 0 and pad[0, 36:72].all()
M = make_att_2d_masks(pad, att)
assert not M[1, :, 36:72].any()            # 没有任何 token 能看到缺失相机的 token
print("通过 ✅")

## 5. 位置编码：`position_ids = cumsum(pad_masks) - 1`

Gemma 用 RoPE（4.02 课），每个 token 需要一个位置号。openpi 的写法：

```py
position_ids = torch.cumsum(pad_masks, dim=1) - 1
```

**padding 不占位置号**：padding 处 cumsum 不增加。所以不管指令有多长、相机是否缺失，有效 token 的位置号总是连续的 0, 1, 2, ...，后缀（动作 token）的位置号紧接着前缀的有效 token 继续往后排。

### 练习 3：验证 padding 不影响有效 token 的输出

同一条指令，分别 padding 到长度 12 和 16，前缀的有效 token 输出应该**完全相同**（这要求位置号和掩码都处理正确）。

In [ ]:
@torch.no_grad()
def prefix_outputs(model, images, ids, msk):
    """返回 VLM 对前缀的输出 (B, L, W)"""
    embs, pad, att = model.embed_prefix(images, ids, msk)
    # TODO: pos = cumsum(pad) - 1；用 model.moe([embs, None], make_att_2d_masks(pad, att), pos) 得到前缀输出
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
@torch.no_grad()
def prefix_outputs(model, images, ids, msk):
    """返回 VLM 对前缀的输出 (B, L, W)"""
    embs, pad, att = model.embed_prefix(images, ids, msk)
    pos = torch.cumsum(pad.long(), 1) - 1
    (out, _), _ = model.moe([embs, None], make_att_2d_masks(pad, att), pos)
    return out

In [ ]:
# ✅ 自测
model.eval()
img1 = imgs[:1]
ids12, msk12 = tok.encode(["去红色方块"], max_len=12)
ids16, msk16 = tok.encode(["去红色方块"], max_len=16)
o12 = prefix_outputs(model, img1, torch.tensor(ids12), torch.tensor(msk12))
o16 = prefix_outputs(model, img1, torch.tensor(ids16), torch.tensor(msk16))
n_valid = 36 + int(msk12.sum())
assert torch.allclose(o12[:, :n_valid], o16[:, :n_valid], atol=1e-5)
print("通过 ✅ padding 的长度不影响有效 token 的计算结果")

---

## 小结

- PaliGemma = SigLIP（图像 → 256 个 token/相机）+ 线性投影 + Gemma（文字 embedding × √d）。
- 图像预处理：`resize_with_pad` 保持长宽比；缺失的相机用 `img_mask` 整组屏蔽。
- π0 状态在后缀（连续 token），π0.5 状态离散化后写进提示词。
- 前缀内部 `att_masks` 全 0（双向），与 VLM 预训练一致；`position_ids = cumsum(pad) - 1`，padding 不占位置。